In [1]:
import os
import duckdb
import boto3
from dotenv import load_dotenv
import pandas as pd

Configuração do Pandas

In [2]:
pd.set_option('display.max_columns', None)

Carregando as variáveis do arquivo .env

In [22]:
load_dotenv()

PROFILE = os.getenv("AWS_PROFILE")
REGION = os.getenv("AWS_REGION")
PREFIX = os.getenv("PROJECT_BUCKET_PREFIX")
RAW_BUCKET = f"{PREFIX}-raw"
BRONZE_BUCKET = f"{PREFIX}-bronze"

Definindo as conexões com o S3 e o DuckDB

In [6]:
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()

con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")

Carregando arquivo do Bucket S3

In [24]:
csv = f"s3://{RAW_BUCKET}/perfil_comparecimento_abstencao/2022/perfil_comparecimento_abstencao_2022_BRASIL.csv"
pqt = f"s3://{BRONZE_BUCKET}/perfil_comparecimento_abstencao/2022/perfil_comparecimento_abstencao_2022_BRASIL.parquet"

In [25]:
raw_file = f"read_csv_auto('{csv}', header=true, delim=';', encoding='latin-1', all_varchar=true)"

#### Leitura do Arquivo Bruto

Verificação rápida da presença de divergências entre o arquivo da camada raw para camada bronze.

In [21]:
con.execute(f"DESCRIBE SELECT * FROM {raw_file}").df()

,column_name,column_type,null,key,default,extra
0,DT_GERACAO,VARCHAR,YES,None,None,None
1,HH_GERACAO,VARCHAR,YES,None,None,None
2,ANO_ELEICAO,VARCHAR,YES,None,None,None
3,NR_TURNO,VARCHAR,YES,None,None,None
4,SG_UF,VARCHAR,YES,None,None,None
5,CD_MUNICIPIO,VARCHAR,YES,None,None,None
6,NM_MUNICIPIO,VARCHAR,YES,None,None,None
7,NR_ZONA,VARCHAR,YES,None,None,None
8,CD_GENERO,VARCHAR,YES,None,None,None
9,DS_GENERO,VARCHAR,YES,None,None,None


In [27]:
con.execute(f"DESCRIBE SELECT * FROM '{pqt}'").df()


,column_name,column_type,null,key,default,extra
0,DT_GERACAO,VARCHAR,YES,None,None,None
1,HH_GERACAO,VARCHAR,YES,None,None,None
2,ANO_ELEICAO,VARCHAR,YES,None,None,None
3,NR_TURNO,VARCHAR,YES,None,None,None
4,SG_UF,VARCHAR,YES,None,None,None
5,CD_MUNICIPIO,VARCHAR,YES,None,None,None
6,NM_MUNICIPIO,VARCHAR,YES,None,None,None
7,NR_ZONA,VARCHAR,YES,None,None,None
8,CD_GENERO,VARCHAR,YES,None,None,None
9,DS_GENERO,VARCHAR,YES,None,None,None


In [30]:
# Cor/Raça do Arquivo CSV na RAW
con.execute(f""" 
    SELECT DISTINCT
        p.CD_COR_RACA,
        P.DS_COR_RACA      
    FROM {raw_file} as p
""").df()

,CD_COR_RACA,DS_COR_RACA
0,-1,NÃO INFORMADO


In [ ]:
# Cor/Raça da SILVER
con.execute(f""" 
    SELECT DISTINCT
        p.CD_COR_RACA,
        P.DS_COR_RACA      
    FROM '{pqt}' as p
""").df()

,CD_COR_RACA,DS_COR_RACA
0,-1,NÃO INFORMADO
